# LLM-DM：批量 Colab 队列

保留已冻结的数据、方法版本和预算，只把手动逐个运行改为有限的串行队列。
默认只准备和预览；先批量运行所有方法的完整 seed-0 pilot，统一人工检查，再批量运行正式 seeds 38–45。
已完成的相同任务会校验后跳过。失败或备份异常会停止，不自动重试，不改变预算。

**已经在原 notebook 完成环境和 Drive 挂载？** 不需要新开运行时；把下面「加载批量入口」的代码复制到原 notebook 即可。
不要让两个 notebook 同时写入同一份 STATE / BACKUPS。运行时回收后先恢复备份，中断任务仍需人工检查。


In [ ]:
from pathlib import Path
import json, re, subprocess, sys, os, platform, hashlib, importlib.metadata

RELEASE_REVISION = "2a40f564dd33d117d322a57746d43fc174180cdf"
UPSTREAM_REVISION = "37269969a0957448d51622e0c083977bc5d260e8"
REPO = Path("/content/llmdm_repo")
UPSTREAM = Path("/content/data-recipes")
if platform.system() != "Linux" or not Path("/content").is_dir():
    raise RuntimeError("请在 Google Colab 中运行此 notebook。")
if not re.fullmatch(r"[0-9a-f]{40}", RELEASE_REVISION):
    raise RuntimeError("发布版本未固定，禁止执行。")

def checkout_exact(url, destination, revision):
    if destination.exists():
        actual = subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip()
        if actual != revision:
            raise RuntimeError(f"{destination} 不是要求的版本；请使用新的运行时，不要覆盖旧目录。")
        if subprocess.check_output(["git", "-C", str(destination), "status", "--porcelain"], text=True).strip():
            raise RuntimeError(f"{destination} 有本地修改，请先检查。")
        return
    subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", url, str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "-c", "core.autocrlf=false", "checkout", "--detach", revision], check=True)

checkout_exact("https://github.com/Kaiyue2003/llm-design-bench.git", REPO, RELEASE_REVISION)
checkout_exact("https://github.com/namkoong-lab/data-recipes.git", UPSTREAM, UPSTREAM_REVISION)
ASSETS = REPO / "experiments/llmdm_forward_v1"
release = json.loads((ASSETS / "release.json").read_text())
print("Release checkout:", RELEASE_REVISION)
print("Frozen code commit:", release["code_commit"])
print("Data manifest:", release["data_manifest_id"])


## 1. 安装环境，保留 Colab 的 CUDA PyTorch

不要运行 data-recipes 的旧版 setup_env.sh，不安装它完整的优化框架依赖。此单元不训练、不调用 oracle。安装后如果 Colab 提示重启，请重启 Python 会话并重新运行准备单元。


In [ ]:
torch_before = importlib.metadata.version("torch")
constraints = Path("/content/llmdm-torch-constraint.txt")
constraints.write_text(f"torch=={torch_before}\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(constraints), "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)
assert importlib.metadata.version("torch") == torch_before, "PyTorch 被更换，请检查环境"
os.environ["MPLCONFIGDIR"] = "/content/llmdm-mpl-cache"
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["free", "-h"], check=True)

# Editable installs may not refresh an already-running notebook kernel.
source_path = str(REPO / "src")
if source_path not in sys.path:
    sys.path.insert(0, source_path)
importlib.invalidate_caches()
import llm_design_bench
assert Path(llm_design_bench.__file__).resolve() == (REPO / "src/llm_design_bench/__init__.py").resolve()
print("Benchmark import:", llm_design_bench.__file__)


## 2. 只读校验真实数据与冻结计划

应看到 logged=454、main visible=184、fixed-1B=26。这里不重新切分、不重新冻结预算，也不加载 oracle checkpoint。


In [ ]:
from llm_design_bench.evaluation.data_manifest import load_data_manifest
from llm_design_bench.evaluation.llmdm_protocol import load_method_plan

for relative, expected_hash in release["artifact_sha256"].items():
    actual_hash = hashlib.sha256((ASSETS / relative).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise RuntimeError(f"发布文件校验失败: {relative}")
bundle = load_data_manifest(ASSETS / "data", data_recipes_root=UPSTREAM)
plan = load_method_plan(ASSETS / "plan.json", bundle)
assert bundle.manifest_id == release["data_manifest_id"]
assert plan["plan_id"] == release["plan_id"]
assert len(bundle.reference_utility) == 454
assert len(bundle.utility) == 184
assert int((bundle.context[:, 0] == 1000).sum()) == 26
assert plan["package_source"]["git_dirty"] is False
print("校验通过：454 logged / 184 main visible / 26 fixed-1B")
print("可选方法:", ", ".join(entry["run_id"] for entry in plan["methods"]))
print("Frozen plan:", plan["plan_id"])


## 3. 挂载 Drive 与恢复备份

本地 /content 是实际运行位置；Drive 保存带 SHA256 校验的不可覆盖归档和启动记录。

这是你授权的 Drive 挂载操作，Google 会要求你选择账户。不要把未审查的 notebook 授予 Drive 权限。此 notebook 的结果路径限定为下面的 benchmark 目录。

恢复只会跳过已完成且校验通过的 seed。被中断的训练不会从中间参数继续；必须检查记录并填写基础设施中断原因，才能按原配置重跑。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, str(REPO / "scripts"))
from colab_support import restore_latest, run_job

STATE = Path("/content/llmdm_run_state")
BACKUPS = Path("/content/drive/MyDrive/llm_design_bench/llmdm_forward_v1") / plan["plan_id"]
BACKUPS.mkdir(parents=True, exist_ok=True)
if not STATE.exists() or not any(STATE.iterdir()):
    if list((BACKUPS / "snapshots").glob("snapshot-*.tar.gz")):
        restored = restore_latest(BACKUPS, STATE)
        print("恢复归档:", restored)
    else:
        STATE.mkdir(parents=True, exist_ok=True)
        print("新的实验状态目录")
else:
    print("继续使用本运行时已有状态；不覆盖")
print("持久备份目录:", BACKUPS)


## 4. 加载批量入口（只预览，不训练）

仅下载固定提交、校验 SHA256 的新运行脚本，放在独立缓存目录；原方法代码和实验状态不会替换。


In [ ]:
"""Load the batch launcher into an already prepared Colab session; never train.

Execute this commit-pinned file in the original notebook's namespace, after
Drive mounting. It leaves the frozen checkout and all existing results intact.
"""

from __future__ import annotations

import hashlib
import importlib.util
import re
import sys
import urllib.request
from pathlib import Path

BATCH_REVISION = "5f9bd5dc208d53f6f22684b76645168977ffd1b0"
BATCH_SHA256 = "459700324f4e71ebc5d5f1dc16a3a5fb4fcf910e0722fcfe1b5ce9efddeef0d9"
SUPPORT_SHA256 = "d607d5893fbf1b70b70d0d23762ba6394b727c620d564637a7463ed2f084b48e"
MAX_SCRIPT_BYTES = 1024 * 1024


def _fetch_launcher(cache_root: Path) -> Path:
    if not re.fullmatch(r"[0-9a-f]{40}", BATCH_REVISION):
        raise ValueError("batch launcher revision must be a full pinned commit")
    if not re.fullmatch(r"[0-9a-f]{64}", BATCH_SHA256):
        raise ValueError("batch launcher must have a pinned SHA256")
    destination = cache_root / BATCH_REVISION / "colab_batch.py"
    if any(path.is_symlink() for path in (cache_root, destination.parent, destination)):
        raise ValueError("launcher cache must not be a symlink")
    if destination.exists():
        with destination.open("rb") as handle:
            source = handle.read(MAX_SCRIPT_BYTES + 1)
    else:
        url = (
            "https://raw.githubusercontent.com/Kaiyue2003/llm-design-bench/"
            f"{BATCH_REVISION}/scripts/colab_batch.py"
        )
        with urllib.request.urlopen(url, timeout=30) as response:
            source = response.read(MAX_SCRIPT_BYTES + 1)
        if len(source) > MAX_SCRIPT_BYTES:
            raise ValueError("downloaded launcher exceeds its size limit")
        if hashlib.sha256(source).hexdigest() != BATCH_SHA256:
            raise ValueError("downloaded batch launcher hash mismatch; nothing loaded")
        destination.parent.mkdir(parents=True, exist_ok=True)
        with destination.open("xb") as handle:
            handle.write(source)
    if len(source) > MAX_SCRIPT_BYTES or hashlib.sha256(source).hexdigest() != BATCH_SHA256:
        raise ValueError("cached batch launcher hash mismatch; nothing loaded")
    return destination


def load_batch_runner(
    repo,
    assets,
    upstream,
    state,
    backups,
    *,
    neural_device="cuda",
    allow_environment_change=False,
):
    """Load checked orchestration code without changing experiment Python files."""
    repo = Path(repo).resolve()
    helper = repo / "scripts/colab_support.py"
    if (
        hashlib.sha256(helper.read_bytes().replace(b"\r\n", b"\n")).hexdigest()
        != SUPPORT_SHA256
    ):
        raise ValueError(
            "original single-job helper changed; do not mix launcher versions"
        )
    helpers = str(helper.parent)
    if helpers not in sys.path:
        sys.path.insert(0, helpers)
    import colab_support

    if Path(colab_support.__file__).resolve() != helper.resolve():
        raise ValueError("another colab_support module is already loaded")
    path = _fetch_launcher(repo.parent / "llmdm_batch_tools")
    name = "llmdm_colab_batch_" + BATCH_REVISION
    specification = importlib.util.spec_from_file_location(name, path)
    if specification is None or specification.loader is None:
        raise ImportError("cannot load the verified batch launcher")
    module = importlib.util.module_from_spec(specification)
    sys.modules[name] = module
    specification.loader.exec_module(module)
    return module.BatchRunner(
        assets=assets,
        data_recipes_root=upstream,
        state=state,
        backups=backups,
        neural_device=neural_device,
        allow_environment_change=allow_environment_change,
    )


if __name__ == "__main__":
    missing = [
        name
        for name in ("REPO", "ASSETS", "UPSTREAM", "STATE", "BACKUPS")
        if name not in globals()
    ]
    if missing:
        raise RuntimeError(
            "先完成原 notebook 的环境、数据校验和 Drive 挂载单元。缺少变量: "
            + ", ".join(missing)
        )
    batch = load_batch_runner(
        globals()["REPO"],
        globals()["ASSETS"],
        globals()["UPSTREAM"],
        globals()["STATE"],
        globals()["BACKUPS"],
        neural_device=globals().get("NEURAL_DEVICE", "cuda"),
        allow_environment_change=globals().get("ALLOW_ENVIRONMENT_CHANGE", False),
    )
    import pandas as pd
    from IPython.display import display

    queue = pd.DataFrame(batch.preview(phase="pilot"))
    display(
        queue[
            ["setting", "run_id", "seed", "device", "status"]
            + (["error"] if "error" in queue else [])
        ]
    )
    print(
        "批量入口加载成功；这里只预览，没有启动训练。complete 会校验后跳过，blocked 需要先检查。"
    )



## 5. 选择队列

第一次保持 `PHASE="pilot"`，检查预览后设置 `RUN_BATCH=True`。不需要填写 seed：pilot 自动为 0，formal 自动为 38–45。

`METHODS=None` 选择全部 19 个方法；也可以填 `["coms", "bdi"]`。默认只跑主实验。若选择 fixed_1b，它需要自己的完整 pilot。


In [ ]:
METHODS = None
SETTINGS = ("multi_scale",)
PHASE = "pilot"
NEURAL_DEVICE = "cuda"
RUN_BATCH = False
CONFIRM_BATCH_PILOTS_REVIEWED = False
ALLOW_ENVIRONMENT_CHANGE = False

batch = load_batch_runner(REPO, ASSETS, UPSTREAM, STATE, BACKUPS,
                          neural_device=NEURAL_DEVICE,
                          allow_environment_change=ALLOW_ENVIRONMENT_CHANGE)
queue = pd.DataFrame(batch.preview(methods=METHODS, settings=SETTINGS, phase=PHASE))
display(queue[["setting", "run_id", "seed", "device", "status"] + (["error"] if "error" in queue else [])])


## 6. 运行整个队列（显式开关）

每个实际任务单独启动子进程并备份；校验后跳过的任务不会重训或重复归档。任一任务失败、环境冲突或备份失败都会停止队列。

pilot 队列不会自动进入 formal。读完下面的报告、确认资源与数值诊断后，再将 PHASE 改为 formal，并显式确认。不要根据 oracle 分数挑预算/seed。

请检查 Drive 剩余空间；全量归档会累积。批处理不延长 Colab 的运行时限制，也不是后台保活任务。


In [ ]:
if not RUN_BATCH:
    print("只预览，尚未训练。确认队列后设置 RUN_BATCH=True，重新运行选择和执行单元。")
else:
    if PHASE == "formal" and not CONFIRM_BATCH_PILOTS_REVIEWED:
        raise RuntimeError("请先检查所有选中方法/设置的 pilot 报告，再显式确认。")
    selected_methods = [x["run_id"] for x in batch.plan["methods"]] if METHODS is None else list(METHODS)
    reviewed = [(setting, run_id) for setting in SETTINGS for run_id in selected_methods] if PHASE == "formal" else []
    batch_results = batch.run(methods=METHODS, settings=SETTINGS, phase=PHASE,
                              reviewed_pilots=reviewed)
    display(pd.DataFrame(batch_results))
    print("队列完成；completed 是本次新完成，skipped 是已验证的已有结果。")


## 7. 统一查看 pilot 诊断与正式覆盖率

pilot 报告校验已有文件并展示训练摘要、时间和显存，不重新训练或查询 oracle。有限的最终诊断不能代替完整训练轨迹；缺失字段要检查，不应当作零成本。

正式实验每个方法/设置必须有 8 个成功 seeds、0 失败、0 缺失才满足覆盖要求。下表不是按 pilot oracle 分数筛选方法的依据。


In [ ]:
pilot_report = pd.DataFrame(batch.pilot_report(methods=METHODS, settings=SETTINGS))
display(pilot_report)
formal_path = STATE / "formal" / "method_seed_summary.csv"
if formal_path.exists():
    formal = pd.read_csv(formal_path)
    fields = ["task_id", "run_id", "successful_runs", "failed_runs", "missing_runs",
              "rank_eligible", "method_seconds_mean"]
    display(formal[[c for c in fields if c in formal]])
print("备份位置:", BACKUPS)
